# Backend Colab pentru Demo Aplicatie

Acest notebook ruleaza un server FastAPI care expune doua modele:
- **VLM** (Qwen2.5-VL): proceseaza email + document, returneaza JSON complet
- **Donut**: proceseaza document, returneaza campuri extrase

Aplicatia locala (Streamlit) trimite cereri catre acest server prin ngrok.

**Setup:** Runtime → T4 GPU. Apoi ruleaza toate celulele in ordine.

In [ ]:
# 1. Install dependencies
!pip install -q transformers accelerate qwen-vl-utils pypdfium2
!pip install -q fastapi uvicorn pyngrok python-multipart pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 36.3 MB/s eta 0:00:00


In [ ]:
# 2. Mount Drive and load models from saved checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile

DRIVE_FOLDER = '/content/drive/My Drive/disertatie'

# Extract Donut model from Drive
donut_zip = os.path.join(DRIVE_FOLDER, 'donut_models_2.zip')
if os.path.exists(donut_zip):
    print(f'Extracting {donut_zip}...')
    with zipfile.ZipFile(donut_zip, 'r') as z:
        z.extractall('.')
    print('Done!')
else:
    print(f'WARNING: {donut_zip} not found.')
    print('You need to upload donut_models.zip to your Drive folder.')

# Verify what we have
for path in ['donut_clean/best_model', 'donut_augmented/best_model']:
    if os.path.exists(path):
        print(f'  OK: {path}')

Mounted at /content/drive
Extracting /content/drive/My Drive/disertatie/donut_models_2.zip...
Done!
  OK: donut_clean/best_model
  OK: donut_augmented/best_model


In [ ]:
# 3. Load both models into GPU memory
import torch
from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    DonutProcessor, VisionEncoderDecoderModel,
)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n')

# Load VLM (Qwen2.5-VL)
print('Loading Qwen2.5-VL...')
VLM_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='eager',
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
print(f'  VLM loaded! GPU usage: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load Donut (augmented version)
print('\nLoading Donut (augmented)...')
DONUT_PATH = 'donut_augmented/best_model'
if not os.path.exists(DONUT_PATH):
    DONUT_PATH = 'donut_clean/best_model'
    print(f'  Using: {DONUT_PATH}')

donut_processor = DonutProcessor.from_pretrained(DONUT_PATH)
donut_model = VisionEncoderDecoderModel.from_pretrained(DONUT_PATH).to('cuda')
donut_model.eval()
print(f'  Donut loaded! Total GPU usage: {torch.cuda.memory_allocated()/1e9:.1f} GB')

GPU: Tesla T4
GPU Memory: 15.6 GB

Loading Qwen2.5-VL...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

  VLM loaded! GPU usage: 7.5 GB

Loading Donut (augmented)...


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

  Donut loaded! Total GPU usage: 8.3 GB


In [ ]:
# 4. Inference functions
import re
import json
from PIL import Image
import pypdfium2 as pdfium
import io
import base64

# ═══════════════════════════════════════════════════════════
# PROMPT SELECTOR — Schimba aici ce prompt sa folosesti
# ═══════════════════════════════════════════════════════════
PROMPT_VERSION = 'strict'  # Options: 'moderate' or 'strict'

# ─── Varianta A: Moderata (mai permisiva) ───
VLM_PROMPT_MODERATE = """You are a document analysis system. You will receive an email from a supplier
and optionally a document image (invoice, quotation, or price list).

Your task is to extract structured information and detect any mismatches between the two sources.

Respond ONLY with a JSON object (no markdown, no backticks, no explanation). The JSON must have:
{
  "intent": one of ["invoice_submission", "quote_offer", "price_increase", "price_validity_confirmation", "other"],
  "email_fields": {"amount": number or null, "currency": string or null, "doc_number": string or null, "date": "YYYY-MM-DD" or null},
  "document_fields": {"amount": number or null, "currency": string or null, "doc_number": string or null, "date": "YYYY-MM-DD" or null},
  "is_consistent": true/false,
  "mismatched_fields": [list of field names that differ]
}

Rules:
- email_fields should only contain values mentioned in the email text
- document_fields should only contain values mentioned in the document
- For "amount": extract monetary totals, not percentages or quantities
- For "date": extract due dates or validity dates in YYYY-MM-DD format
- mismatched_fields should contain names like "amount", "date", etc.
- A field is a mismatch only if BOTH sources have a value AND they differ
"""

# ─── Varianta B+: Stricta cu separare secventiala a surselor ───
VLM_PROMPT_STRICT = """You are a STRICT reconciliation system for supplier correspondence.

You are given TWO INDEPENDENT SOURCES about the same transaction:
  (A) the EMAIL TEXT  — written by the sender in the message body
  (B) the ATTACHED DOCUMENT — an image (invoice, quote, price list, etc.)

These are SEPARATE sources. Your job is to extract fields from EACH source on its
own, and ONLY THEN compare them. One source must NEVER influence the other. The
whole point of the task is to detect when the two sources DISAGREE — which is
impossible if you let values leak from one into the other.

CRITICAL OUTPUT FORMAT — Respond with ONLY valid JSON, no other text:
{
  "intent": "invoice_submission" | "quote_offer" | "price_increase" | "price_validity_confirmation" | "other",
  "email_fields": {"amount": <number|null>, "currency": <string|null>, "doc_number": <string|null>, "date": <string|null>},
  "document_fields": {"amount": <number|null>, "currency": <string|null>, "doc_number": <string|null>, "date": <string|null>},
  "is_consistent": <boolean>,
  "mismatched_fields": <array of strings>
}

═══════════════════════════════════════════════
MANDATORY PROCEDURE — follow these steps IN ORDER (think internally, output only JSON)
═══════════════════════════════════════════════
STEP 1 — EMAIL FIRST (image hidden):
  Read ONLY the email text. Pretend the attached document DOES NOT EXIST and you
  cannot see any image. Fill "email_fields" using ONLY words present in the email
  text. Any field not literally written in the email text → null.

STEP 2 — DOCUMENT SECOND (email hidden):
  Now read ONLY the document image. Pretend the email text DOES NOT EXIST. Fill
  "document_fields" using ONLY what is visibly printed in the image. Any field not
  visible in the image → null.

STEP 3 — COMPARE (reconcile the two completed sets):
  Only now, compare the two sets field by field to produce "mismatched_fields"
  and "is_consistent". Do NOT change any extracted value during this step.

═══════════════════════════════════════════════
RULE 0: SOURCE ISOLATION (THE MOST IMPORTANT RULE)
═══════════════════════════════════════════════
- Every value in "email_fields" MUST be traceable to a literal span in the EMAIL TEXT.
- Every value in "document_fields" MUST be traceable to something visible in the IMAGE.
- If you cannot point to WHERE in that specific source a value came from, it is null.
- NEVER copy a value from one source into the other to "fill a gap".
- If a field appears in only one source, the other source's field stays null. That
  is expected and correct — it is NOT a mismatch (see RULE 5).
- Two sources showing the SAME value is a coincidence worth reporting (it means they
  agree); it is NOT permission to copy a value you only saw in one source.

═══════════════════════════════════════════════
RULE 1: INTENT (MUST BE ONE OF FIVE EXACT VALUES)
═══════════════════════════════════════════════
"intent" MUST be EXACTLY one of:
- "invoice_submission" — sender is sending an invoice for payment
- "quote_offer" — sender is providing a price quotation/offer
- "price_increase" — sender announces new prices or price changes (e.g., "3% increase", "new 2026 prices")
- "price_validity_confirmation" — sender confirms existing prices remain valid
- "other" — none of the above
DO NOT output any other string. DO NOT describe the task in this field.

═══════════════════════════════════════════════
RULE 2: AMOUNT
═══════════════════════════════════════════════
- "amount" = a MONETARY TOTAL (e.g., 12500.00). Output as a NUMBER, not a string.
- Percentages ("3% increase") are NOT amounts → null.
- Quantities ("5 items", "20 units") are NOT amounts → null.
- If no monetary total is explicitly stated in that source, amount = null.

═══════════════════════════════════════════════
RULE 3: CURRENCY
═══════════════════════════════════════════════
- Exactly one of: "USD", "EUR", "GBP", "RON", "CHF".
- Must be EXPLICITLY present in that source (code, or symbol $, €, £, or word).
- Never infer currency from context, country, or the other source → null if absent.

═══════════════════════════════════════════════
RULE 4: DATE
═══════════════════════════════════════════════
- Output format: "YYYY-MM-DD".
- "31.12.2026" → "2026-12-31"; "December 15, 2025" → "2025-12-15".
- Extract the relevant date in that source (due / validity / effective date).

═══════════════════════════════════════════════
RULE 5: MISMATCH DETECTION
═══════════════════════════════════════════════
- "mismatched_fields" = field NAMES whose values DIFFER and are non-null in BOTH sources.
- Allowed names ONLY: "amount", "currency", "doc_number", "date".
- A field is a mismatch ONLY IF email_fields[field] != null AND document_fields[field] != null AND they differ.
- If EITHER source has null for that field → it is NEVER a mismatch.
- "is_consistent" = true if "mismatched_fields" is empty, else false.

═══════════════════════════════════════════════
SELF-CHECK BEFORE OUTPUT (do this internally)
═══════════════════════════════════════════════
1. For each non-null value in email_fields: can I quote the email text it came from? If not → set it to null.
2. For each non-null value in document_fields: is it actually visible in the image? If not → set it to null.
3. Did any value end up identical in both sources only because I copied it? If yes → re-derive it from each source independently.
4. Does every entry in mismatched_fields have non-null values in BOTH sources? Remove any that do not.

═══════════════════════════════════════════════
NEGATIVE EXAMPLES (DO NOT DO THIS)
═══════════════════════════════════════════════
WRONG: "intent": "Extract fields from the document"
RIGHT: "intent": "price_increase"

WRONG: "amount": 3            (email says "3% increase")
RIGHT: "amount": null

WRONG: email_fields.currency = "EUR" because the DOCUMENT shows EUR (email text never mentions currency)
RIGHT: email_fields.currency = null

WRONG: email_fields.amount = 2111.73 because the invoice image shows 2111.73 (email text has no amount)
RIGHT: email_fields.amount = null

WRONG: "mismatched_fields": ["amount"] when email_fields.amount is null
RIGHT: "mismatched_fields": []   (no mismatch possible if one side is null)

WRONG: "mismatched_fields": ["$2,111.73", "Effective: 2025-01-03"]
RIGHT: "mismatched_fields": ["amount", "date"]

Now follow the MANDATORY PROCEDURE and respond with ONLY valid JSON.
"""

# ─── Select active prompt ───
if PROMPT_VERSION == 'strict':
    VLM_SYSTEM_PROMPT = VLM_PROMPT_STRICT
    print('Using STRICT prompt')
else:
    VLM_SYSTEM_PROMPT = VLM_PROMPT_MODERATE
    print('Using MODERATE prompt')

# ═══════════════════════════════════════════════════════════
# Helpers
# ═══════════════════════════════════════════════════════════

def load_image_from_bytes(image_bytes):
    """Load image from bytes (base64 decoded)."""
    return Image.open(io.BytesIO(image_bytes)).convert('RGB')

def load_pdf_from_bytes(pdf_bytes):
    """Convert PDF first page to PIL image."""
    pdf = pdfium.PdfDocument(pdf_bytes)
    page = pdf[0]
    bitmap = page.render(scale=200/72)
    img = bitmap.to_pil()
    pdf.close()
    return img

def query_vlm(email_subject, email_body, image=None, max_tokens=600):
    """Run VLM inference."""
    user_prompt = f'Email subject: {email_subject}\n\nEmail body:\n{email_body}\n'
    if image:
        user_prompt += '\nThe attached document image is provided. Extract fields from both.'
    else:
        user_prompt += '\nNo document is attached. Set all document_fields to null.'

    content = []
    if image:
        max_dim = 1024
        if max(image.size) > max_dim:
            ratio = max_dim / max(image.size)
            image = image.resize((int(image.width * ratio), int(image.height * ratio)))
        content.append({'type': 'image', 'image': image})
    content.append({'type': 'text', 'text': user_prompt})

    messages = [
        {'role': 'system', 'content': VLM_SYSTEM_PROMPT},
        {'role': 'user', 'content': content},
    ]
    text_input = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if image:
        inputs = vlm_processor(text=[text_input], images=[image], padding=True, return_tensors='pt').to(vlm_model.device)
    else:
        inputs = vlm_processor(text=[text_input], padding=True, return_tensors='pt').to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, temperature=0.0)
    gen_ids = output_ids[:, inputs['input_ids'].shape[1]:]
    response = vlm_processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

    # Parse JSON
    text = response.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except: pass
        return None

def query_donut(image, image_size=(360, 480)):
    """Run Donut inference on document image."""
    image = image.resize(image_size)
    pixel_values = donut_processor(image, return_tensors='pt').pixel_values.to('cuda')

    task_prompt = '<s_extract>'
    decoder_input_ids = donut_processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to('cuda')

    with torch.no_grad():
        outputs = donut_model.generate(
            pixel_values, decoder_input_ids=decoder_input_ids, max_length=128,
            num_beams=1, pad_token_id=donut_processor.tokenizer.pad_token_id,
            eos_token_id=donut_processor.tokenizer.eos_token_id,
        )
    raw = donut_processor.tokenizer.decode(outputs[0], skip_special_tokens=False)

    result = {}
    for field in ['amount', 'currency', 'doc_number', 'date']:
        m = re.search(f'<s_{field}>(.*?)</s_{field}>', raw)
        if m:
            val = re.sub(r'</?s_\w+>', '', m.group(1)).strip()
            result[field] = val if val else None
        else:
            result[field] = None
    return result

print('Inference functions ready!')

Using STRICT prompt
Inference functions ready!


In [ ]:
# 5. FastAPI server
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title='Disertatie Backend', version='1.0')
app.add_middleware(
    CORSMiddleware, allow_origins=['*'], allow_credentials=True,
    allow_methods=['*'], allow_headers=['*'],
)

class HealthResponse(BaseModel):
    status: str
    vlm_loaded: bool
    donut_loaded: bool
    gpu: str

@app.get('/health', response_model=HealthResponse)
def health():
    return {
        'status': 'ok',
        'vlm_loaded': vlm_model is not None,
        'donut_loaded': donut_model is not None,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    }

@app.post('/vlm')
async def vlm_endpoint(
    subject: str = Form(...),
    body: str = Form(...),
    image: Optional[UploadFile] = File(None),
):
    """VLM endpoint: processes email + optional document."""
    img = None
    if image is not None:
        content = await image.read()
        if image.filename.lower().endswith('.pdf'):
            img = load_pdf_from_bytes(content)
        else:
            img = load_image_from_bytes(content)

    result = query_vlm(subject, body, img)
    if result is None:
        raise HTTPException(status_code=500, detail='VLM parsing failed')
    return result

@app.post('/donut')
async def donut_endpoint(image: UploadFile = File(...)):
    """Donut endpoint: processes document image only."""
    content = await image.read()
    if image.filename.lower().endswith('.pdf'):
        img = load_pdf_from_bytes(content)
    else:
        img = load_image_from_bytes(content)
    return query_donut(img)

print('FastAPI app defined!')

FastAPI app defined!


In [ ]:
# 6. Setup ngrok and start server
# Get your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from threading import Thread

NGROK_AUTHTOKEN = '3DUAeSDvrU4w5y8Tx1L1iQd0Vni_88Wq7jRsdJRSg891hhfFJ'  # PASTE YOUR TOKEN

ngrok.set_auth_token(NGROK_AUTHTOKEN)
nest_asyncio.apply()

# Kill old tunnels
ngrok.kill()

# Start tunnel
public_url = ngrok.connect(8000).public_url
print(f'\n{"="*60}')
print(f'PUBLIC URL: {public_url}')
print(f'{"="*60}')
print(f'\nCopy this URL into your local Streamlit app config:')
print(f'  COLAB_BACKEND_URL = "{public_url}"')
print(f'\nTest from terminal:')
print(f'  curl {public_url}/health')
print()

# Start server in thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

thread = Thread(target=run_server, daemon=True)
thread.start()
print('Server is running!')
print('Keep this notebook open during demo.')


PUBLIC URL: https://kept-rescuer-crisping.ngrok-free.dev

Copy this URL into your local Streamlit app config:
  COLAB_BACKEND_URL = "https://kept-rescuer-crisping.ngrok-free.dev"

Test from terminal:
  curl https://kept-rescuer-crisping.ngrok-free.dev/health

Server is running!
Keep this notebook open during demo.


INFO:     Started server process [648]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


In [ ]:
# 7. Test endpoint (optional)
import requests

# Test health
r = requests.get(f'{public_url}/health')
print('Health:', r.json())

In [ ]:
# 8. Keep alive (optional - prevents Colab from disconnecting)
import time
from IPython.display import display, Javascript

print('Server is running. Backend is ready for inference requests.')
print(f'URL: {public_url}')
print('\nTo stop: Runtime → Disconnect and delete runtime')

# Optional: prevent Colab disconnect by running a periodic ping
while True:
    try:
        time.sleep(60)
        # Just check that server is still up
        r = requests.get(f'{public_url}/health', timeout=5)
        # print(f'.', end='', flush=True)
    except KeyboardInterrupt:
        print('\nStopped.')
        break
    except Exception as e:
        print(f'\nError: {e}')
        break